In [0]:
#BRONZE TO SILVER
import yaml
from pyspark.sql.types import NumericType, StringType, DateType, TimestampType, DecimalType, IntegerType
from pyspark.sql.functions import col, when, lit, regexp_replace, date_format, from_utc_timestamp, current_timestamp, year, month, dayofmonth
from delta.tables import DeltaTable


dbutils.widgets.text("NB_Storage_Account","")
storage_account = dbutils.widgets.get("NB_Storage_Account")

config_path = f"abfss://config@{storage_account}.dfs.core.windows.net/databricks_config.yaml"
config = dbutils.fs.head(config_path)
config_parse = yaml.safe_load(config)

dbutils.widgets.text("NB_Year", "")
dbutils.widgets.text("NB_Month", "")
param_year = str(dbutils.widgets.get("NB_Year"))
param_month = str(dbutils.widgets.get("NB_Month"))
print(config_path)
print(param_year)
print(param_month)

def table_has_schema(table_name):
    try:
        df = spark.table(table_name)
        return len(df.columns) > 0
    except Exception:
        return False

for table_name in config_parse["tables"]:

        full_path =  f"<catalog>.bronze.{table_name}"

        if not table_has_schema(full_path):
                print(f"SKIPPED: {table_name} Year={param_year} Month={param_month} Bronze Table Not Exist Yet")
                continue
        
        df_to_clean_table = spark.table(full_path).filter(
                        (col("Year") == param_year) &
                        (col("Month") == param_month)
        )

        if df_to_clean_table.isEmpty():
                print(f"SKIPPED: {table_name} Year={param_year} Month={param_month} — no data in bronze")
                continue

        #Getting Yaml Table Configuration
        #get columns to be DROP
        ToDropColumns = config_parse["tables"][table_name].get("Drop_Columns", [])
        #get Columns to be Flag if NUll or Zero
        ToFlagIfNullOrZeroColumns = config_parse["tables"][table_name].get("Flag_If_Null_Or_Zero",[])
        #get columns to be NULL TO ZERO
        ToNullToZeroColumns = config_parse["tables"][table_name].get("Null_To_Zero",[])
        #get columns to be NULL TO NA
        ToNullToNAColumns = config_parse["tables"][table_name].get("Null_To_NA",[])
        #get columns to be MASKED
        ToMaskedColumns = config_parse["tables"][table_name].get("Mask_Columns",[])
        #get columns to be Force Schema
        ToForceSchemaColumns = config_parse["tables"][table_name].get("Schema_Columns",[])
        #get columns to be RENAME
        ToRenameColumns = config_parse["tables"][table_name].get("Rename_Columns",[])
        #get merge keys
        Merge_keys = config_parse["tables"][table_name].get("Merge_Keys",[]) 


        #DROP COLUMNS
        #check if existing columns in yaml is still existing in dataframe
        existing_columns = [drop_col for drop_col in ToDropColumns if drop_col in df_to_clean_table.columns]
        #run drop columns
        df_to_clean_table = df_to_clean_table.drop(*existing_columns)

        #Schema Columns
        for schema_col, schema_type in ToForceSchemaColumns.items():
                if schema_col in df_to_clean_table.columns:
                        df_to_clean_table = df_to_clean_table.withColumn(schema_col, col(schema_col).cast(schema_type))
                        # print("schema")


        #FLAG NULL OR ZERO
        for flag_rule in ToFlagIfNullOrZeroColumns:
                flag_columns = flag_rule["Column"]
                flag_when_column = flag_rule.get("If_Column")
                flag_when_status = flag_rule.get("If_Status", [])


                if flag_columns in df_to_clean_table.columns:
                        is_numeric = isinstance(df_to_clean_table.schema[flag_columns].dataType, NumericType)
                        # if when_column and when_status is not null
                        if is_numeric:
                                if flag_when_column is not None and flag_when_status is not None:
                                        df_to_clean_table = df_to_clean_table.withColumn(
                                                "Remarks",
                                                when((col(flag_when_column).isin(flag_when_status)) & (col(flag_columns)== 0), lit(f"{flag_columns} Is Zero"))
                                                .otherwise(lit(None))
                                        )
                        else:
                                df_to_clean_table = df_to_clean_table.withColumn(
                                        "Remarks",
                                        when(col(flag_columns).isNull(), lit(f"{flag_columns} Is Missing"))
                                        .otherwise(lit(None))
                                                )
                                
        #NULL TO ZERO COLUMNS
        #check if numeric columns in yaml
        numeric_columns = [numeric_col for numeric_col in ToNullToZeroColumns if isinstance(df_to_clean_table.schema[numeric_col].dataType, NumericType)]
        #Change NULL TO ZERO
        dict_nulltozero = {numeric_col: 0 for numeric_col in numeric_columns}
        #RUN NULL TO ZERO
        df_to_clean_table = df_to_clean_table.fillna(dict_nulltozero)
        # print("nulltozero")

        #NULL TO NA COLUMNS
        #check if string columns in yaml
        string_columns = [NA_col for NA_col in ToNullToNAColumns if isinstance(df_to_clean_table.schema[NA_col].dataType, StringType)]
        #Change NULL TO NA
        dict_nulltona = {NA_col: "NA" for NA_col in string_columns}
        #run NULL TO NA
        df_to_clean_table = df_to_clean_table.fillna(dict_nulltona)
        # print("nulltoNA")

        #MASKED COLUMNS
        for mask_rule in ToMaskedColumns:
                mask_column = mask_rule["Column"]
                mask_when_column = mask_rule.get("If_Column")
                mask_when_status = mask_rule.get("If_Null", False)

                if mask_column in df_to_clean_table.columns:
                        if mask_when_column is not None and mask_when_status == True:
                                df_to_clean_table = df_to_clean_table.withColumn(mask_column,when((col(mask_when_column)== "") | (col(mask_when_column)=="NA"), col(mask_column)).otherwise(lit("********")))
                        else:
                                df_to_clean_table = df_to_clean_table.withColumn(mask_column, lit("********"))

        #RENAMING COLUMNS
        for old_name, new_name in ToRenameColumns.items():
                df_to_clean_table = df_to_clean_table.withColumnRenamed(old_name, new_name)

        # add modified date
        df_to_clean_table = df_to_clean_table.withColumn("ModifiedDate",  from_utc_timestamp(current_timestamp(), "Asia/Manila"))

        #REMOVE DUPLICATES
        df_to_clean_table = df_to_clean_table.dropDuplicates(Merge_keys)

        if spark.catalog.tableExists(f"<catalog>.silver.{table_name}"):
                merge_key_conditions = []
                for k in Merge_keys:
                        condition = f"target.`{k}` = source.`{k}`"
                        merge_key_conditions.append(condition)
                key_condition = " AND ".join(merge_key_conditions)

                partition_condition = f"target.Year = '{param_year}' AND target.Month = '{param_month}'"

                full_condition = f"{key_condition} AND {partition_condition}"
                (DeltaTable.forName(spark, f"<catalog>.silver.{table_name}").alias("target")
                        .merge(df_to_clean_table.alias("source"), full_condition)
                        .whenMatchedUpdateAll()
                        .whenNotMatchedInsertAll()
                        .execute())
                print(f"Saved Silver Data For: {table_name}, Year={param_year}, Month={param_month}")
        else:
                df_to_clean_table.write \
                        .format("delta") \
                        .option("path", f"abfss://silver@{storage_account}.dfs.core.windows.net/sales/{table_name}/") \
                        .partitionBy("Year", "Month", "Day") \
                        .saveAsTable(f"<catalog>.silver.{table_name}")
                print(f"Saved Silver Data For: {table_name}, Year={param_year}, Month={param_month}")